# Model Selection Benchmark

Benchmarks **7 VLM models** on **20 CORD receipt images** with:
- **Operational metrics:** Load time, VRAM, Avg inference time, JSON parse success rate
- **Accuracy metrics:** Field Exact Match, Field Token F1, Menu Item F1

Each model runs in its own cell so a CUDA crash doesn't kill the entire session.

**Runtime:** Google Colab T4 GPU (15 GB VRAM)

---

### Multi-Session Workflow (if runtime disconnects)

Each model saves its result to `results/model_{name}.json`. If your session dies:

1. **Download** the `results/model_*.json` files before the session ends (or use Google Drive)
2. **Restart** a new Colab session, run Cells 1-4 (setup only, no GPU needed yet)
3. **Upload** your saved JSON files back to `results/` folder
4. **Skip** to the "Combine & Compare" section (Cell 13+) — it loads from files, not memory

You can run 2-3 models per session across multiple days and still compare all of them at the end.

In [1]:
# Cell 1: Install dependencies
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils[decord] datasets pillow
!pip install -q PyMuPDF opencv-python-headless matplotlib pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.1 MB/s eta 0:00:00


In [2]:
# Cell 2: Clone repo + imports + GPU check
import os, sys

REPO_DIR = "/content/VLM-PDF-Text_Extractor"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RahulReadd/VLM-PDF-Text_Extractor.git
os.chdir(REPO_DIR)
sys.path.insert(0, ".")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {total_vram:.1f} GB")
else:
    raise RuntimeError("No GPU found. Enable GPU runtime: Runtime > Change runtime type > T4 GPU")

Cloning into 'VLM-PDF-Text_Extractor'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 71 (delta 31), reused 61 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 53.19 KiB | 4.09 MiB/s, done.
Resolving deltas: 100% (31/31), done.
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:

# Cell 3: Load 20 CORD images + ground truths
from datasets import load_dataset

EVAL_SIZE = 20

cord = load_dataset("naver-clova-ix/cord-v2", split="test")

# Sample evenly across the dataset for diversity
import numpy as np
indices = np.linspace(0, len(cord) - 1, EVAL_SIZE, dtype=int).tolist()

test_images = [cord[i]["image"].convert("RGB") for i in indices]
ground_truths = [cord[i]["ground_truth"] for i in indices]

print(f"Loaded {len(test_images)} test images from CORD dataset")
print(f"Image sizes: {[img.size for img in test_images[:3]]} ...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00004-7dbbe248962764(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

...

data/test-00000-of-00001-9c204eb3f4e1179(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Loaded 20 test images from CORD dataset
Image sizes: [(432, 648), (2304, 4096), (576, 864)] ...


In [4]:
indices

[0, 5, 10, 15, 20, 26, 31, 36, 41, 46, 52, 57, 62, 67, 72, 78, 83, 88, 93, 99]

In [5]:
from PIL import Image

MAX_PIXELS = 512 * 28 * 28  # ~401K — matches VLM vision encoder budget

def preprocess_for_vlm(img: Image.Image, max_pixels: int = MAX_PIXELS) -> Image.Image:
    """Resize image so total pixels stay within the VLM's vision encoder budget."""
    w, h = img.size
    total = w * h
    if total > max_pixels:
        scale = (max_pixels / total) ** 0.5
        new_w = max(28, int(w * scale))
        new_h = max(28, int(h * scale))
        img = img.resize((new_w, new_h), Image.LANCZOS)
    return img

test_images = [preprocess_for_vlm(img) for img in test_images]

In [6]:
# Cell 4: Define benchmark_with_accuracy helper + receipt prompt
import json
import time
from pathlib import Path
from statistics import mean

from src.benchmark import benchmark_single_model, save_report_csv
from app.evaluate import evaluate_single, parse_cord_ground_truth

RECEIPT_PROMPT = """Analyze this receipt image. Extract all information and return a JSON object with these fields:

{
  "menu": [
    {"nm": "item name", "cnt": "quantity", "price": "unit price"}
  ],
  "sub_total": {
    "subtotal_price": "...",
    "tax_price": "...",
    "discount_price": "..."
  },
  "total": {
    "total_price": "...",
    "cashprice": "...",
    "changeprice": "..."
  }
}

Rules:
- Return ONLY valid JSON, no extra text or explanation.
- Use null for any field you cannot find in the image.
- Include ALL menu items visible on the receipt.
"""


def benchmark_with_accuracy(model_name, images, prompts, ground_truths):
    """Run operational benchmark + accuracy evaluation for one model."""
    report = benchmark_single_model(model_name, images, prompts)

    accuracy_scores = []
    for result, gt_raw in zip(report.per_image_results, ground_truths):
        gt_parsed = parse_cord_ground_truth(gt_raw)
        scores = evaluate_single(result.output_json, gt_parsed)
        accuracy_scores.append(scores)

    n = len(accuracy_scores)
    avg_field_em = mean(s["field_em"] for s in accuracy_scores) if n else 0.0
    avg_field_f1 = mean(s["field_f1"] for s in accuracy_scores) if n else 0.0
    avg_menu_f1 = mean(s["menu_f1"] for s in accuracy_scores) if n else 0.0

    summary = {
        "model_name": model_name,
        "model_id": report.model_id,
        "load_time_s": report.load_time_s,
        "vram_allocated_mb": report.vram_after_load_mb[0],
        "vram_reserved_mb": report.vram_after_load_mb[1],
        "avg_inference_s": round(report.avg_inference_s, 2),
        "json_success_rate": round(report.json_success_rate, 3),
        "avg_field_em": round(avg_field_em, 3),
        "avg_field_f1": round(avg_field_f1, 3),
        "avg_menu_f1": round(avg_menu_f1, 3),
        "n_images": n,
        "per_image_accuracy": accuracy_scores,
    }

    print(f"\n{'─'*50}")
    print(f"  {model_name} — Summary")
    print(f"{'─'*50}")
    print(f"  Load:      {summary['load_time_s']:.1f}s")
    print(f"  VRAM:      {summary['vram_allocated_mb']:.0f} MB alloc / {summary['vram_reserved_mb']:.0f} MB reserved")
    print(f"  Avg Inf:   {summary['avg_inference_s']:.2f}s")
    print(f"  JSON %:    {summary['json_success_rate']*100:.0f}%")
    print(f"  Field EM:  {summary['avg_field_em']:.3f}")
    print(f"  Field F1:  {summary['avg_field_f1']:.3f}")
    print(f"  Menu F1:   {summary['avg_menu_f1']:.3f}")

    return summary


def save_model_result(summary, out_dir="results"):
    """Persist one model's summary to disk + Google Drive (if mounted)."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"model_{summary['model_name']}.json")
    serializable = {k: v for k, v in summary.items() if k != "per_image_accuracy"}
    with open(path, "w") as f:
        json.dump(serializable, f, indent=2)
    print(f"  Saved to {path}")

    # Also save to Google Drive if mounted (survives runtime restarts)
    drive_dir = "/content/drive/MyDrive/VLM_model_results"
    if os.path.exists("/content/drive/MyDrive"):
        os.makedirs(drive_dir, exist_ok=True)
        drive_path = os.path.join(drive_dir, f"model_{summary['model_name']}.json")
        with open(drive_path, "w") as f:
            json.dump(serializable, f, indent=2)
        print(f"  Backed up to Google Drive: {drive_path}")


# Build prompts list (one per image)
prompts = [RECEIPT_PROMPT] * len(test_images)

print(f"Helper functions ready. Will benchmark on {len(test_images)} images.")
print(f"Results will be saved to results/ (and Google Drive if mounted)")

Helper functions ready. Will benchmark on 20 images.
Results will be saved to results/ (and Google Drive if mounted)


In [7]:
# Cell 5: HuggingFace login (required for Llama-3.2 Vision)
# If you don't have a token or haven't accepted the Llama 3.2 license,
# the llama32-11b-4bit cell will fail gracefully — other models are unaffected.
#
# Accept the license at: https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct

from huggingface_hub import login
try:
    login()  # Will prompt for token if not already logged in
    print("HuggingFace login successful.")
except Exception as e:
    print(f"HuggingFace login skipped: {e}")
    print("Llama-3.2 Vision will not be available. Other models are unaffected.")

HuggingFace login successful.


In [8]:
# Cell 5b: Mount Google Drive (optional but recommended)
# Results are backed up here automatically — survives runtime restarts.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted. Results will be backed up to: /content/drive/MyDrive/VLM_model_results/")
except Exception:
    print("Google Drive not available. Results will only be saved locally.")

Mounted at /content/drive
Google Drive mounted. Results will be backed up to: /content/drive/MyDrive/VLM_model_results/


---
## Model Benchmarks

Each model runs in its own cell. If a CUDA error kills one model,
**restart the runtime** and re-run Cells 1-4, then skip to the next model cell.

Order: smallest VRAM first, largest last.

### Model 1: Qwen3-VL-2B (2B, ~4 GB VRAM)

In [9]:
# Cell 7: qwen3-vl-2b
summary_qwen3_2b = benchmark_with_accuracy("qwen3-vl-2b", test_images, prompts, ground_truths)
save_model_result(summary_qwen3_2b)


  Loading: qwen3-vl-2b (Qwen/Qwen3-VL-2B-Instruct)
  dtype=torch.float16, quant=None


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

  Loaded in 77.2s | VRAM: 4058 MB allocated, 4078 MB reserved
  Image 1/20: 13.4s | JSON valid: True | VRAM: 4067 MB
  Image 2/20: 9.5s | JSON valid: True | VRAM: 4067 MB
  Image 3/20: 8.3s | JSON valid: True | VRAM: 4067 MB
  Image 4/20: 11.2s | JSON valid: True | VRAM: 4067 MB
  Image 5/20: 17.8s | JSON valid: True | VRAM: 4067 MB
  Image 6/20: 16.0s | JSON valid: True | VRAM: 4067 MB
  Image 7/20: 11.6s | JSON valid: True | VRAM: 4067 MB
  Image 8/20: 11.1s | JSON valid: True | VRAM: 4067 MB
  Image 9/20: 30.4s | JSON valid: True | VRAM: 4067 MB
  Image 10/20: 10.9s | JSON valid: True | VRAM: 4067 MB
  Image 11/20: 14.5s | JSON valid: True | VRAM: 4067 MB
  Image 12/20: 10.7s | JSON valid: True | VRAM: 4067 MB
  Image 13/20: 10.1s | JSON valid: True | VRAM: 4067 MB
  Image 14/20: 9.6s | JSON valid: True | VRAM: 4067 MB
  Image 15/20: 9.4s | JSON valid: True | VRAM: 4067 MB
  Image 16/20: 10.5s | JSON valid: True | VRAM: 4067 MB
  Image 17/20: 10.2s | JSON valid: True | VRAM: 4067 MB

In [17]:
summary_qwen3_2b

{'model_name': 'qwen3-vl-2b',
 'model_id': 'Qwen/Qwen3-VL-2B-Instruct',
 'load_time_s': 77.15,
 'vram_allocated_mb': 4058.0,
 'vram_reserved_mb': 4078.0,
 'avg_inference_s': 13.0,
 'json_success_rate': 1.0,
 'avg_field_em': 0.492,
 'avg_field_f1': 0.525,
 'avg_menu_f1': 0.768,
 'n_images': 20,
 'per_image_accuracy': [{'json_valid': True,
   'field_em': 0.6666666666666666,
   'field_f1': 0.6666666666666666,
   'menu_f1': 0.0},
  {'json_valid': True,
   'field_em': 0.3333333333333333,
   'field_f1': 0.3333333333333333,
   'menu_f1': 1.0},
  {'json_valid': True, 'field_em': 1.0, 'field_f1': 1.0, 'menu_f1': 1.0},
  {'json_valid': True, 'field_em': 1.0, 'field_f1': 1.0, 'menu_f1': 1.0},
  {'json_valid': True, 'field_em': 0.0, 'field_f1': 0.0, 'menu_f1': 1.0},
  {'json_valid': True,
   'field_em': 0.0,
   'field_f1': 0.6666666666666666,
   'menu_f1': 0.0},
  {'json_valid': True,
   'field_em': 0.3333333333333333,
   'field_f1': 0.3333333333333333,
   'menu_f1': 1.0},
  {'json_valid': True, '

### Model 3: Qwen2.5-VL-3B (3B, ~7 GB VRAM)

In [10]:
# Cell 8: qwen25-vl-3b
summary_qwen25 = benchmark_with_accuracy("qwen25-vl-3b", test_images, prompts, ground_truths)
save_model_result(summary_qwen25)


  Loading: qwen25-vl-3b (Qwen/Qwen2.5-VL-3B-Instruct)
  dtype=torch.float16, quant=None


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Loaded in 161.0s | VRAM: 7171 MB allocated, 7246 MB reserved
  Image 1/20: 10.9s | JSON valid: True | VRAM: 7171 MB
  Image 2/20: 10.8s | JSON valid: True | VRAM: 7171 MB
  Image 3/20: 9.7s | JSON valid: True | VRAM: 7171 MB
  Image 4/20: 12.7s | JSON valid: True | VRAM: 7171 MB
  Image 5/20: 21.1s | JSON valid: True | VRAM: 7171 MB
  Image 6/20: 17.9s | JSON valid: True | VRAM: 7171 MB
  Image 7/20: 13.9s | JSON valid: True | VRAM: 7171 MB
  Image 8/20: 12.9s | JSON valid: True | VRAM: 7171 MB
  Image 9/20: 34.8s | JSON valid: True | VRAM: 7171 MB
  Image 10/20: 13.3s | JSON valid: True | VRAM: 7171 MB
  Image 11/20: 13.0s | JSON valid: True | VRAM: 7171 MB
  Image 12/20: 12.4s | JSON valid: True | VRAM: 7171 MB
  Image 13/20: 13.0s | JSON valid: True | VRAM: 7171 MB
  Image 14/20: 10.2s | JSON valid: True | VRAM: 7171 MB
  Image 15/20: 10.2s | JSON valid: True | VRAM: 7171 MB
  Image 16/20: 12.6s | JSON valid: True | VRAM: 7171 MB
  Image 17/20: 10.8s | JSON valid: True | VRAM: 717

In [ ]:
summary_qwen25

{'model_name': 'qwen25-vl-3b',
 'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
 'load_time_s': 135.91,
 'vram_allocated_mb': 7161.4,
 'vram_reserved_mb': 7234.0,
 'avg_inference_s': 11.95,
 'json_success_rate': 1.0,
 'avg_field_em': 0.658,
 'avg_field_f1': 0.658,
 'avg_menu_f1': 0.859,
 'n_images': 20,
 'per_image_accuracy': [{'json_valid': True,
   'field_em': 0.6666666666666666,
   'field_f1': 0.6666666666666666,
   'menu_f1': 1.0},
  {'json_valid': True,
   'field_em': 0.3333333333333333,
   'field_f1': 0.3333333333333333,
   'menu_f1': 1.0},
  {'json_valid': True, 'field_em': 1.0, 'field_f1': 1.0, 'menu_f1': 1.0},
  {'json_valid': True, 'field_em': 1.0, 'field_f1': 1.0, 'menu_f1': 1.0},
  {'json_valid': True,
   'field_em': 0.3333333333333333,
   'field_f1': 0.3333333333333333,
   'menu_f1': 1.0},
  {'json_valid': True, 'field_em': 1.0, 'field_f1': 1.0, 'menu_f1': 0.0},
  {'json_valid': True,
   'field_em': 0.3333333333333333,
   'field_f1': 0.3333333333333333,
   'menu_f1': 1.0},
  {'

### Model 4: Qwen3-VL-4B (4B, ~8.5 GB VRAM)

In [ ]:
# Cell 9: qwen3-vl-4b
summary_qwen3_4b = benchmark_with_accuracy("qwen3-vl-4b", test_images, prompts, ground_truths)
save_model_result(summary_qwen3_4b)


  Loading: qwen3-vl-4b (Qwen/Qwen3-VL-4B-Instruct)
  dtype=torch.float16, quant=None


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

  Loaded in 194.4s | VRAM: 8481 MB allocated, 8516 MB reserved
  Image 1/20: 8.9s | JSON valid: True | VRAM: 8481 MB
  Image 2/20: 9.8s | JSON valid: True | VRAM: 8481 MB
  Image 3/20: 9.4s | JSON valid: True | VRAM: 8481 MB
  Image 4/20: 10.5s | JSON valid: True | VRAM: 8481 MB
  Image 5/20: 18.6s | JSON valid: True | VRAM: 8481 MB
  Image 6/20: 15.7s | JSON valid: True | VRAM: 8481 MB
  Image 7/20: 11.7s | JSON valid: True | VRAM: 8481 MB
  Image 8/20: 10.8s | JSON valid: True | VRAM: 8481 MB
  Image 9/20: 31.3s | JSON valid: True | VRAM: 8481 MB
  Image 10/20: 11.4s | JSON valid: True | VRAM: 8481 MB
  Image 11/20: 11.7s | JSON valid: True | VRAM: 8481 MB
  Image 12/20: 10.3s | JSON valid: True | VRAM: 8481 MB
  Image 13/20: 11.4s | JSON valid: True | VRAM: 8481 MB
  Image 14/20: 9.6s | JSON valid: True | VRAM: 8481 MB
  Image 15/20: 8.9s | JSON valid: True | VRAM: 8481 MB
  Image 16/20: 10.9s | JSON valid: True | VRAM: 8481 MB
  Image 17/20: 9.8s | JSON valid: True | VRAM: 8481 MB


### Model 5: InternVL 3.5-8B 4-bit (~6-7 GB VRAM)

In [ ]:
# Cell 10: internvl35-8b-4bit
summary_internvl = benchmark_with_accuracy("internvl35-8b-4bit", test_images, prompts, ground_truths)
save_model_result(summary_internvl)


  Loading: internvl35-8b-4bit (OpenGVLab/InternVL3_5-8B-HF)
  dtype=torch.float16, quant=4bit


config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

17.8s | JSON valid: True | VRAM: 6312 MB
  Image 19/20: 

[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


16.0s | JSON valid: True | VRAM: 6312 MB
  Image 20/20: 

[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


30.3s | JSON valid: True | VRAM: 6312 MB

──────────────────────────────────────────────────
  internvl35-8b-4bit — Summary
──────────────────────────────────────────────────
  Load:      558.6s
  VRAM:      6312 MB alloc / 7322 MB reserved
  Avg Inf:   18.69s
  JSON %:    100%
  Field EM:  0.492
  Field F1:  0.525
  Menu F1:   0.838
  Saved to results/model_internvl35-8b-4bit.json
  Backed up to Google Drive: /content/drive/MyDrive/VLM_model_results/model_internvl35-8b-4bit.json


### Model 6: Pixtral-12B 4-bit (~8.5 GB VRAM)

In [ ]:
# Cell 11: pixtral-12b-4bit
summary_pixtral = benchmark_with_accuracy("pixtral-12b-4bit", test_images, prompts, ground_truths)
save_model_result(summary_pixtral)


  Loading: pixtral-12b-4bit (mistral-community/pixtral-12b)
  dtype=torch.float16, quant=4bit


config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

...

22.4s | JSON valid: True | VRAM: 8705 MB
  Image 19/20: 

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


17.7s | JSON valid: True | VRAM: 8705 MB
  Image 20/20: 

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


42.7s | JSON valid: True | VRAM: 8705 MB

──────────────────────────────────────────────────
  pixtral-12b-4bit — Summary
──────────────────────────────────────────────────
  Load:      916.3s
  VRAM:      8705 MB alloc / 8726 MB reserved
  Avg Inf:   23.79s
  JSON %:    100%
  Field EM:  0.208
  Field F1:  0.208
  Menu F1:   0.735
  Saved to results/model_pixtral-12b-4bit.json
  Backed up to Google Drive: /content/drive/MyDrive/VLM_model_results/model_pixtral-12b-4bit.json


### Model 7: Llama-3.2-11B Vision 4-bit (~8 GB VRAM)

Requires HuggingFace login (Cell 5) and accepting the [Llama 3.2 Community License](https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct).

In [ ]:
# Cell 12: llama32-11b-4bit
try:
    summary_llama = benchmark_with_accuracy("llama32-11b-4bit", test_images, prompts, ground_truths)
    save_model_result(summary_llama)
except Exception as e:
    print(f"\nLlama-3.2 Vision failed: {e}")
    print("Ensure you have: 1) Accepted the license on HuggingFace  2) Logged in via Cell 5")


  Loading: llama32-11b-4bit (meta-llama/Llama-3.2-11B-Vision-Instruct)
  dtype=torch.float16, quant=4bit


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

  Loaded in 330.6s | VRAM: 7311 MB allocated, 7348 MB reserved
  Image 1/20: 19.4s | JSON valid: True | VRAM: 7320 MB
  Image 2/20: 14.6s | JSON valid: True | VRAM: 7320 MB
  Image 3/20: 12.9s | JSON valid: True | VRAM: 7320 MB
  Image 4/20: 14.5s | JSON valid: True | VRAM: 7320 MB
  Image 5/20: 21.1s | JSON valid: False | VRAM: 7320 MB
  Image 6/20: 33.0s | JSON valid: True | VRAM: 7320 MB
  Image 7/20: 19.2s | JSON valid: False | VRAM: 7320 MB
  Image 8/20: 15.5s | JSON valid: True | VRAM: 7320 MB
  Image 9/20: 31.4s | JSON valid: True | VRAM: 7320 MB
  Image 10/20: 13.7s | JSON valid: True | VRAM: 7320 MB
  Image 11/20: 22.7s | JSON valid: True | VRAM: 7320 MB
  Image 12/20: 14.4s | JSON valid: False | VRAM: 7320 MB
  Image 13/20: 14.6s | JSON valid: True | VRAM: 7320 MB
  Image 14/20: 21.8s | JSON valid: True | VRAM: 7320 MB
  Image 15/20: 12.5s | JSON valid: True | VRAM: 7320 MB
  Image 16/20: 14.5s | JSON valid: False | VRAM: 7320 MB
  Image 17/20: 16.9s | JSON valid: False | VRA

### Save checkpoint: Download results so far

Run this cell after each model (or batch of models) to download a ZIP of all results collected so far. This is your **insurance** against runtime disconnects.

In [11]:
# Cell 12a: Download all results collected so far
#
# Downloads a ZIP of all model results — save this to your local machine.
# If your runtime dies, you can upload these back in the recovery cell.

import glob, zipfile, os
from datetime import datetime

result_files = sorted(glob.glob("results/model_*.json"))
if not result_files:
    print("No model results saved yet. Run model cells first.")
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_name = f"model_results_{ts}.zip"
    with zipfile.ZipFile(zip_name, "w") as zf:
        for f in result_files:
            zf.write(f, os.path.basename(f))
    print(f"Zipped {len(result_files)} result files → {zip_name}")

    try:
        from google.colab import files
        files.download(zip_name)
        print("Download started! Save this zip for recovery.")
    except ImportError:
        print(f"Not in Colab. ZIP saved at: {os.path.abspath(zip_name)}")

Zipped 2 result files → model_results_20260413_093732.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started! Save this zip for recovery.


---
## Results: Combine & Compare

In [18]:
# Cell 12b: Recovery — restore results from previous sessions
#
# Run this ONLY if you're resuming from a crashed/disconnected session.
# Skip this cell if you just finished running the model cells above.
#
# Two recovery options:
#   Option A: Auto-recover from Google Drive (if mounted in Cell 5b)
#   Option B: Upload JSON files from your local machine via browser dialog

import os, json, shutil

RESULTS_DIR = "results"
DRIVE_DIR = "/content/drive/MyDrive/VLM_model_results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Option A: Auto-recover from Google Drive ──
recovered = 0
if os.path.exists(DRIVE_DIR):
    for fname in os.listdir(DRIVE_DIR):
        if fname.startswith("model_") and fname.endswith(".json"):
            src = os.path.join(DRIVE_DIR, fname)
            dst = os.path.join(RESULTS_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                recovered += 1
                print(f"  Recovered from Drive: {fname}")
            else:
                print(f"  Already exists locally: {fname}")
    if recovered:
        print(f"\nRestored {recovered} result file(s) from Google Drive.")
    else:
        print("All Drive results already present locally.")
else:
    print("Google Drive not mounted — skipping auto-recovery.")

# ── Option B: Upload from local machine ──
# A browser dialog will open — select your model_*.json files (or the ZIP).
# Click "Cancel" in the dialog if you don't need to upload anything.

# try:
#     from google.colab import files
#     print("\n📂 Upload dialog opening... Select your model_*.json files (or cancel to skip)")
#     uploaded = files.upload()

#     for fname, content in uploaded.items():
#         if fname.endswith(".zip"):
#             import zipfile, io
#             with zipfile.ZipFile(io.BytesIO(content)) as zf:
#                 for member in zf.namelist():
#                     if member.startswith("model_") and member.endswith(".json"):
#                         zf.extract(member, RESULTS_DIR)
#                         print(f"  Extracted from ZIP: {member}")
#         elif fname.startswith("model_") and fname.endswith(".json"):
#             dst = os.path.join(RESULTS_DIR, fname)
#             with open(dst, "wb") as f:
#                 f.write(content)
#             print(f"  Uploaded: {fname} → {dst}")
#         else:
#             print(f"  Skipped (not a model result): {fname}")
# except ImportError:
#     print("Not running in Colab — upload not available. Place files in results/ manually.")
# except Exception as e:
#     print(f"Upload cancelled or skipped: {e}")

# ── Show all available results ──
import glob
existing = sorted(glob.glob(f"{RESULTS_DIR}/model_*.json"))
print(f"\n{'='*50}")
print(f"  Total model results available: {len(existing)}")
print(f"{'='*50}")
for f in existing:
    with open(f) as fh:
        data = json.load(fh)
    print(f"  {os.path.basename(f):35s}  →  {data.get('model_id', '?')}")

  Already exists locally: model_internvl35-8b-4bit.json
  Already exists locally: model_qwen3-vl-4b.json
  Already exists locally: model_pixtral-12b-4bit.json
  Already exists locally: model_llama32-11b-4bit.json
  Already exists locally: model_qwen25-vl-3b.json
  Already exists locally: model_qwen3-vl-2b.json
All Drive results already present locally.

  Total model results available: 6
  model_internvl35-8b-4bit.json        →  OpenGVLab/InternVL3_5-8B-HF
  model_llama32-11b-4bit.json          →  meta-llama/Llama-3.2-11B-Vision-Instruct
  model_pixtral-12b-4bit.json          →  mistral-community/pixtral-12b
  model_qwen25-vl-3b.json              →  Qwen/Qwen2.5-VL-3B-Instruct
  model_qwen3-vl-2b.json               →  Qwen/Qwen3-VL-2B-Instruct
  model_qwen3-vl-4b.json               →  Qwen/Qwen3-VL-4B-Instruct


In [19]:
# Cell 13: Combine all saved model results into a DataFrame
import pandas as pd
import glob

result_files = sorted(glob.glob("results/model_*.json"))
print(f"Found {len(result_files)} model result files:\n")

rows = []
for f in result_files:
    with open(f) as fh:
        rows.append(json.load(fh))
    print(f"  {f}")

df = pd.DataFrame(rows)
df = df.set_index("model_name")

print(f"\nLoaded results for {len(df)} models.")

Found 6 model result files:

  results/model_internvl35-8b-4bit.json
  results/model_llama32-11b-4bit.json
  results/model_pixtral-12b-4bit.json
  results/model_qwen25-vl-3b.json
  results/model_qwen3-vl-2b.json
  results/model_qwen3-vl-4b.json

Loaded results for 6 models.


In [20]:
# Cell 14: Final comparison table
display_cols = [
    "model_id", "load_time_s", "vram_allocated_mb", "vram_reserved_mb",
    "avg_inference_s", "json_success_rate",
    "avg_field_em", "avg_field_f1", "avg_menu_f1",
]

print("=" * 120)
print("  FULL MODEL COMPARISON")
print("=" * 120)
print()

display_df = df[display_cols].copy()
display_df.columns = [
    "Model ID", "Load (s)", "VRAM Alloc (MB)", "VRAM Res (MB)",
    "Avg Inf (s)", "JSON %",
    "Field EM", "Field F1", "Menu F1",
]

# Format percentages
display_df["JSON %"] = (display_df["JSON %"] * 100).round(0).astype(int).astype(str) + "%"

print(display_df.to_string())
print()

  FULL MODEL COMPARISON

                                                    Model ID  Load (s)  VRAM Alloc (MB)  VRAM Res (MB)  Avg Inf (s) JSON %  Field EM  Field F1  Menu F1
model_name                                                                                                                                             
internvl35-8b-4bit               OpenGVLab/InternVL3_5-8B-HF    558.56           6312.3         7322.0        18.69   100%     0.492     0.525    0.838
llama32-11b-4bit    meta-llama/Llama-3.2-11B-Vision-Instruct    330.61           7311.4         7348.0        19.20    70%     0.000     0.000    0.421
pixtral-12b-4bit               mistral-community/pixtral-12b    916.34           8705.4         8726.0        23.79   100%     0.208     0.208    0.735
qwen25-vl-3b                     Qwen/Qwen2.5-VL-3B-Instruct    161.05           7171.3         7246.0        14.31   100%     0.658     0.658    0.859
qwen3-vl-2b                        Qwen/Qwen3-VL-2B-Instruct   

In [21]:
# Cell 15: Weighted scoring & recommendation
#
# Weights reflect what matters for a document extraction pipeline:
#   - Accuracy is king (0.40): wrong extractions are useless
#   - Speed matters (0.25): throughput for batch processing
#   - VRAM efficiency (0.20): headroom for high-res documents
#   - JSON reliability (0.15): must produce parseable output

WEIGHTS = {
    "accuracy": 0.40,
    "speed": 0.25,
    "vram": 0.20,
    "json": 0.15,
}

scored = df.copy()

# Accuracy composite: average of Field F1 and Menu F1
scored["accuracy_score"] = (scored["avg_field_f1"] + scored["avg_menu_f1"]) / 2

# Speed: normalize so fastest = 1.0 (lower inference time is better)
min_inf = scored["avg_inference_s"].min()
scored["speed_score"] = min_inf / scored["avg_inference_s"]

# VRAM: normalize so lowest = 1.0 (lower VRAM is better)
min_vram = scored["vram_allocated_mb"].min()
scored["vram_score"] = min_vram / scored["vram_allocated_mb"]

# JSON reliability (already 0-1)
scored["json_score"] = scored["json_success_rate"]

# Weighted total
scored["weighted_score"] = (
    WEIGHTS["accuracy"] * scored["accuracy_score"]
    + WEIGHTS["speed"] * scored["speed_score"]
    + WEIGHTS["vram"] * scored["vram_score"]
    + WEIGHTS["json"] * scored["json_score"]
)

# Sort by weighted score descending
ranking = scored[["accuracy_score", "speed_score", "vram_score", "json_score", "weighted_score"]].copy()
ranking = ranking.round(3).sort_values("weighted_score", ascending=False)

print("=" * 90)
print("  WEIGHTED MODEL RANKING")
print(f"  Weights: Accuracy={WEIGHTS['accuracy']}, Speed={WEIGHTS['speed']}, VRAM={WEIGHTS['vram']}, JSON={WEIGHTS['json']}")
print("=" * 90)
print()
print(ranking.to_string())

winner = ranking.index[0]
print(f"\n{'*'*60}")
print(f"  RECOMMENDED MODEL: {winner}")
print(f"  Weighted Score: {ranking.loc[winner, 'weighted_score']:.3f}")
print(f"{'*'*60}")
print(f"\nRationale:")
print(f"  - Accuracy score:  {ranking.loc[winner, 'accuracy_score']:.3f}")
print(f"  - Speed score:     {ranking.loc[winner, 'speed_score']:.3f}")
print(f"  - VRAM score:      {ranking.loc[winner, 'vram_score']:.3f}")
print(f"  - JSON score:      {ranking.loc[winner, 'json_score']:.3f}")

  WEIGHTED MODEL RANKING
  Weights: Accuracy=0.4, Speed=0.25, VRAM=0.2, JSON=0.15

                    accuracy_score  speed_score  vram_score  json_score  weighted_score
model_name                                                                             
qwen3-vl-2b                  0.647        0.987       1.000         1.0           0.855
qwen25-vl-3b                 0.758        0.897       0.566         1.0           0.791
qwen3-vl-4b                  0.704        1.000       0.479         1.0           0.778
internvl35-8b-4bit           0.682        0.686       0.643         1.0           0.723
pixtral-12b-4bit             0.472        0.539       0.466         1.0           0.567
llama32-11b-4bit             0.210        0.668       0.555         0.7           0.467

************************************************************
  RECOMMENDED MODEL: qwen3-vl-2b
  Weighted Score: 0.855
************************************************************

Rationale:
  - Accuracy score:  

In [22]:
# Cell 16: Save results to CSV
os.makedirs("results", exist_ok=True)

# Full comparison CSV
csv_path = "results/eval_summary.csv"
export_df = scored[[
    "model_id", "load_time_s", "vram_allocated_mb", "vram_reserved_mb",
    "avg_inference_s", "json_success_rate",
    "avg_field_em", "avg_field_f1", "avg_menu_f1",
    "accuracy_score", "speed_score", "vram_score", "json_score", "weighted_score",
]].sort_values("weighted_score", ascending=False)

export_df.to_csv(csv_path)
print(f"Saved full results to: {csv_path}")

# Also save ranking
ranking_path = "results/model_ranking.csv"
ranking.to_csv(ranking_path)
print(f"Saved ranking to: {ranking_path}")

print(f"\nDone! {len(export_df)} models compared on {EVAL_SIZE} CORD images.")

Saved full results to: results/eval_summary.csv
Saved ranking to: results/model_ranking.csv

Done! 6 models compared on 20 CORD images.


---
## Use Case Recommendations

Based on the benchmark results above:

| Use Case | Recommended Model | Why |
|----------|-------------------|-----|
| **General document extraction** | Winner from Cell 15 | Best overall weighted score |
| **Fastest throughput (batch)** | Lowest Avg Inf(s) | Minimize per-document latency |
| **Lowest VRAM (multi-model)** | Florence-2-large | Only ~2 GB, leaves room for other tasks |
| **Best accuracy (quality-first)** | Highest Field F1 + Menu F1 | When correctness matters most |
| **Reasoning-heavy tasks** | Llama-3.2-11B or Qwen3-VL-4B | Larger models for complex layouts |